# 1.1 — Selección de Variables (Feature Selection)
**Proyecto:** Predicción de Riesgo de Abandono de Vivienda y Gentrificación en Sonora  
**Contexto:** El objetivo de esta libreta es argumentar y documentar **qué variables entran al modelo y por qué**, garantizando reproducibilidad en las etapas supervisadas y no supervisadas.

## 0. Setup del Proyecto

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Rutas a los datos crudos ──────────────────────────────────────────────────
RUTA_CENSO  = "../data/raw/censo__inegi_2020.csv"
RUTA_CONAPO = "../data/raw/conapo_2020.xlsx"
RUTA_DENUE  = "../data/raw/denue_sonora_2020.csv"

# ── Municipios de las 6 ciudades más grandes de Sonora ────────────────────────
# Cajeme=018, Guaymas=029, Hermosillo=030, Navojoa=042, Nogales=043, S.L.R.C.=055
MUNICIPIOS_SONORA = ['018', '029', '030', '042', '043', '055']

print("Setup completado. Librerías y rutas definidas.")
print(f"Municipios objetivo: {MUNICIPIOS_SONORA}")

Setup completado. Librerías y rutas definidas.
Municipios objetivo: ['018', '029', '030', '042', '043', '055']


---
## 1. Carga de los Datasets

Cargamos los tres archivos fuente. El Censo se lee con `dtype=str` para respetar las claves geográficas (CVE_AGEB, MUN, LOC) como cadenas de texto con ceros a la izquierda.

### 1.1 Censo INEGI 2020

In [2]:
# dtype=str es OBLIGATORIO: preserva ceros a la izquierda en claves geográficas
df_censo = pd.read_csv(RUTA_CENSO, dtype=str, low_memory=False)

print(f"Shape original: {df_censo.shape[0]:,} filas × {df_censo.shape[1]} columnas")
print(f"\nPrimeras columnas (identificadores geográficos):")
display(df_censo[['ENTIDAD','NOM_ENT','MUN','NOM_MUN','LOC','AGEB','MZA']].head())
print(f"\nTotal de columnas: {df_censo.columns.tolist()}")

Shape original: 71,015 filas × 230 columnas

Primeras columnas (identificadores geográficos):


,ENTIDAD,NOM_ENT,MUN,NOM_MUN,LOC,AGEB,MZA
0,26,Sonora,000,Total de la entidad Sonora,0000,0000,000
1,26,Sonora,001,Aconchi,0000,0000,000
2,26,Sonora,001,Aconchi,0001,0000,000
3,26,Sonora,001,Aconchi,0001,0055,000
4,26,Sonora,001,Aconchi,0001,0055,001



Total de columnas: ['ENTIDAD', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'NOM_LOC', 'AGEB', 'MZA', 'POBTOT', 'POBFEM', 'POBMAS', 'P_0A2', 'P_0A2_F', 'P_0A2_M', 'P_3YMAS', 'P_3YMAS_F', 'P_3YMAS_M', 'P_5YMAS', 'P_5YMAS_F', 'P_5YMAS_M', 'P_12YMAS', 'P_12YMAS_F', 'P_12YMAS_M', 'P_15YMAS', 'P_15YMAS_F', 'P_15YMAS_M', 'P_18YMAS', 'P_18YMAS_F', 'P_18YMAS_M', 'P_3A5', 'P_3A5_F', 'P_3A5_M', 'P_6A11', 'P_6A11_F', 'P_6A11_M', 'P_8A14', 'P_8A14_F', 'P_8A14_M', 'P_12A14', 'P_12A14_F', 'P_12A14_M', 'P_15A17', 'P_15A17_F', 'P_15A17_M', 'P_18A24', 'P_18A24_F', 'P_18A24_M', 'P_15A49_F', 'P_60YMAS', 'P_60YMAS_F', 'P_60YMAS_M', 'REL_H_M', 'POB0_14', 'POB15_64', 'POB65_MAS', 'PROM_HNV', 'PNACENT', 'PNACENT_F', 'PNACENT_M', 'PNACOE', 'PNACOE_F', 'PNACOE_M', 'PRES2015', 'PRES2015_F', 'PRES2015_M', 'PRESOE15', 'PRESOE15_F', 'PRESOE15_M', 'P3YM_HLI', 'P3YM_HLI_F', 'P3YM_HLI_M', 'P3HLINHE', 'P3HLINHE_F', 'P3HLINHE_M', 'P3HLI_HE', 'P3HLI_HE_F', 'P3HLI_HE_M', 'P5_HLI', 'P5_HLI_NHE', 'P5_HLI_HE', 'PHOG_IND', 'POB_AFRO

In [3]:
# Filtrar únicamente AGEBs urbanas de los 6 municipios objetivo.
# Reglas del Marco Geoestadístico INEGI:
#   MZA == '000'  → registro resumen del AGEB (no manzana individual)
#   LOC != '0000' → excluir totales de municipio
#   AGEB != '0000'→ excluir totales de localidad
df_censo_6c = df_censo[
    (df_censo['MUN'].str.zfill(3).isin(MUNICIPIOS_SONORA)) &
    (df_censo['LOC'] != '0000') &
    (df_censo['AGEB'] != '0000') &
    (df_censo['MZA'] == '000')
].copy()

# Construir CVE_AGEB de 13 dígitos: ENTIDAD(2) + MUN(3) + LOC(4) + AGEB(4)
df_censo_6c['CVE_AGEB'] = (
    df_censo_6c['ENTIDAD'].str.zfill(2) +
    df_censo_6c['MUN'].str.zfill(3) +
    df_censo_6c['LOC'].str.zfill(4) +
    df_censo_6c['AGEB'].str.zfill(4)
)

print(f"AGEBs en 6 ciudades: {df_censo_6c.shape[0]:,} filas × {df_censo_6c.shape[1]} columnas")
print(f"\nMunicipios encontrados:")
print(df_censo_6c[['MUN','NOM_MUN']].drop_duplicates().sort_values('MUN').to_string(index=False))
print(f"\nVerificación CVE_AGEB (debe ser string de 13 dígitos):")
print(df_censo_6c['CVE_AGEB'].head(3).tolist())
print(f"Longitudes únicas: {df_censo_6c['CVE_AGEB'].str.len().unique()} (todos deben ser 13)")

AGEBs en 6 ciudades: 1,817 filas × 231 columnas

Municipios encontrados:
MUN               NOM_MUN
018                Cajeme
029               Guaymas
030            Hermosillo
042               Navojoa
043               Nogales
055 San Luis Río Colorado

Verificación CVE_AGEB (debe ser string de 13 dígitos):
['2601800010531', '2601800010546', '2601800010550']
Longitudes únicas: [13] (todos deben ser 13)


### 1.2 CONAPO 2020 — Índice de Marginación

In [4]:
df_conapo = pd.read_excel(RUTA_CONAPO, dtype=str)

print(f"Shape: {df_conapo.shape[0]:,} filas × {df_conapo.shape[1]} columnas")
print(f"\nColumnas disponibles:")
print(df_conapo.columns.tolist())
print(f"\nPrimeras filas:")
display(df_conapo.head())

Shape: 2,096 filas × 23 columnas

Columnas disponibles:
['CVE_AGEB', 'ENT', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'NOM_LOC', 'AGEB', 'POB_TOT', 'P6A14NAE', 'SBASC', 'PSDSS', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'OVHAC', 'OVSREF', 'OVSINT', 'OSCEL', 'IM_2020', 'GM_2020', 'IMN_2020']

Primeras filas:


,CVE_AGEB,ENT,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,AGEB,POB_TOT,P6A14NAE,...,OVSEE,OVSAE,OVPT,OVHAC,OVSREF,OVSINT,OSCEL,IM_2020,GM_2020,IMN_2020
0,2600100010055,26,Sonora,001,Aconchi,0001,Aconchi,0055,764,2.85714285714286,...,0.130890052356021,0,0,25.261780104712,2.74869109947644,59.4240837696335,7.85340314136126,122.290232787068,Bajo,0.955285445227713
1,260010001006A,26,Sonora,001,Aconchi,0001,Aconchi,006A,658,6.02409638554217,...,0.151975683890578,0,0,14.1337386018237,3.19148936170213,52.2796352583587,9.11854103343465,121.59987718759,Bajo,0.949892646136707
2,2600100010110,26,Sonora,001,Aconchi,0001,Aconchi,0110,82,0,...,0,1.21951219512195,0,56.0975609756098,0,87.8048780487805,1.21951219512195,121.872595208357,Bajo,0.952023017058022
3,260010001013A,26,Sonora,001,Aconchi,0001,Aconchi,013A,72,0,...,0,0,0,72.2222222222222,8.33333333333333,93.0555555555556,4.16666666666667,121.219751022561,Bajo,0.946923243065619
4,2600200010512,26,Sonora,002,Agua Prieta,0001,Agua Prieta,0512,1176,2.36686390532544,...,0.683176771989752,0,0.170794192997438,24.5943637916311,3.75747224594364,27.8394534585824,6.40478223740393,120.402235976437,Medio,0.940537122056452


### 1.3 DENUE Sonora 2020 — Directorio de Unidades Económicas

In [5]:
df_denue = pd.read_csv(RUTA_DENUE, dtype=str, encoding='latin-1', low_memory=False)

# El CSV de DENUE usa nombres de columna largos en español con acentos y espacios.
# Renombramos a nombres cortos estándar para facilitar el trabajo posterior.
DENUE_RENAME = {
    'ID':                                           'id',
    'Nombre de la Unidad Económica':                'nom_estab',
    'Código de la clase de actividad SCIAN':        'codigo_act',
    'Nombre de clase de la actividad':              'nombre_act',
    'Descripcion estrato personal ocupado':         'per_ocu',
    'Clave entidad':                                'cve_ent',
    'Entidad federativa':                           'entidad',
    'Clave municipio':                              'cve_mun',
    'Municipio':                                    'municipio',
    'Clave localidad':                              'cve_loc',
    'Localidad':                                    'localidad',
    'Área geoestadística básica ':                  'ageb',   # nota: tiene espacio al final
    'Manzana':                                      'manzana',
    'Latitud':                                      'latitud',
    'Longitud':                                     'longitud',
}
df_denue = df_denue.rename(columns=DENUE_RENAME)

print(f"Shape: {df_denue.shape[0]:,} filas × {df_denue.shape[1]} columnas")
print(f"\nColumnas después de renombrar:")
print(df_denue.columns.tolist())
print(f"\nPrimeras filas (columnas clave):")
display(df_denue[['id','nom_estab','codigo_act','nombre_act','cve_ent','cve_mun','cve_loc','ageb']].head())

Shape: 13,794 filas × 42 columnas

Columnas después de renombrar:
['id', 'Clee', 'nom_estab', 'Razón social', 'codigo_act', 'nombre_act', 'per_ocu', 'Tipo de vialidad', 'Nombre de la vialidad', 'Tipo de entre vialidad 1', 'Nombre de entre vialidad 1', 'Tipo de entre vialidad 2', 'Nombre de entre vialidad 2', 'Tipo de entre vialidad 3', 'Nombre de entre vialidad 3', 'Número exterior o kilómetro', 'Letra exterior', 'Edificio', 'Edificio Piso', 'Número interior', 'Letra interior', 'Tipo de asentamiento humano', 'Nombre de asentamiento humano', 'Tipo centro comercial', 'Corredor industrial, centro comercial o mercado público', 'Número de local', 'Código Postal', 'cve_ent', 'entidad', 'cve_mun', 'municipio', 'cve_loc', 'localidad', 'ageb', 'manzana', 'Número de teléfono', 'Correo electrónico', 'Sitio en Internet', 'Tipo de establecimiento', 'latitud', 'longitud', 'Fecha de incorporación al DENUE']

Primeras filas (columnas clave):


,id,nom_estab,codigo_act,nombre_act,cve_ent,cve_mun,cve_loc,ageb
0,9872427,5 H BAZAR,466410,Comercio al por menor de artículos usados,26,018,0001,1281
1,11300897,ACOSTA BAZAR,466410,Comercio al por menor de artículos usados,26,030,0343,4882
2,11277502,ALMACEN,466410,Comercio al por menor de artículos usados,26,030,0001,5397
3,10071826,AMAZOOM BZR,466410,Comercio al por menor de artículos usados,26,055,0001,1800
4,9960673,ANGEL,466410,Comercio al por menor de artículos usados,26,043,0001,038A


---
## 2. Exploración Básica de Estructura

Verificamos dimensiones, tipos de datos y primeras observaciones para entender la estructura de cada fuente antes de seleccionar variables.

In [6]:
print("=" * 60)
print("CENSO INEGI 2020 — Resumen")
print("=" * 60)
print(f"Shape (6 ciudades, nivel AGEB): {df_censo_6c.shape}")
print(f"\nMunicipios presentes: {df_censo_6c['NOM_MUN'].unique()}")
print(f"\nColumnas de vivienda (contienen 'VIV'):")
print([c for c in df_censo_6c.columns if 'VIV' in c])
print(f"\nColumnas de bienes en el hogar (contienen 'VPH'):")
print([c for c in df_censo_6c.columns if 'VPH' in c][:15], "... (y más)")

CENSO INEGI 2020 — Resumen
Shape (6 ciudades, nivel AGEB): (1817, 231)

Municipios presentes: <StringArray>
[               'Cajeme',               'Guaymas',            'Hermosillo',
               'Navojoa',               'Nogales', 'San Luis Río Colorado']
Length: 6, dtype: str

Columnas de vivienda (contienen 'VIV'):
['VIVTOT', 'TVIVHAB', 'TVIVPAR', 'VIVPAR_HAB', 'VIVPARH_CV', 'TVIVPARHAB', 'VIVPAR_DES', 'VIVPAR_UT', 'OCUPVIVPAR']

Columnas de bienes en el hogar (contienen 'VPH'):
['VPH_PISODT', 'VPH_PISOTI', 'VPH_1DOR', 'VPH_2YMASD', 'VPH_1CUART', 'VPH_2CUART', 'VPH_3YMASC', 'VPH_C_ELEC', 'VPH_S_ELEC', 'VPH_AGUADV', 'VPH_AEASP', 'VPH_AGUAFV', 'VPH_TINACO', 'VPH_CISTER', 'VPH_EXCSA'] ... (y más)


In [7]:
print("=" * 60)
print("CONAPO 2020 — Resumen")
print("=" * 60)
print(f"Shape: {df_conapo.shape}")
print(f"\nColumnas:")
print(df_conapo.columns.tolist())
display(df_conapo.head(3))

print("\n" + "=" * 60)
print("DENUE Sonora 2020 — Resumen")
print("=" * 60)
print(f"Shape: {df_denue.shape}")
print(f"\nDistribución de códigos SCIAN de interés:")
scian_interes = ['522110', '722515', '531210', '522292', '468211', '468212']
for cod in scian_interes:
    n = (df_denue['codigo_act'] == cod).sum()
    nombre = df_denue.loc[df_denue['codigo_act'] == cod, 'nombre_act'].iloc[0] if n > 0 else "N/A"
    print(f"  {cod} | {n:>5} registros | {nombre[:50]}")

CONAPO 2020 — Resumen
Shape: (2096, 23)

Columnas:
['CVE_AGEB', 'ENT', 'NOM_ENT', 'MUN', 'NOM_MUN', 'LOC', 'NOM_LOC', 'AGEB', 'POB_TOT', 'P6A14NAE', 'SBASC', 'PSDSS', 'OVSDE', 'OVSEE', 'OVSAE', 'OVPT', 'OVHAC', 'OVSREF', 'OVSINT', 'OSCEL', 'IM_2020', 'GM_2020', 'IMN_2020']


,CVE_AGEB,ENT,NOM_ENT,MUN,NOM_MUN,LOC,NOM_LOC,AGEB,POB_TOT,P6A14NAE,...,OVSEE,OVSAE,OVPT,OVHAC,OVSREF,OVSINT,OSCEL,IM_2020,GM_2020,IMN_2020
0,2600100010055,26,Sonora,001,Aconchi,0001,Aconchi,0055,764,2.85714285714286,...,0.130890052356021,0,0,25.261780104712,2.74869109947644,59.4240837696335,7.85340314136126,122.290232787068,Bajo,0.955285445227713
1,260010001006A,26,Sonora,001,Aconchi,0001,Aconchi,006A,658,6.02409638554217,...,0.151975683890578,0,0,14.1337386018237,3.19148936170213,52.2796352583587,9.11854103343465,121.59987718759,Bajo,0.949892646136707
2,2600100010110,26,Sonora,001,Aconchi,0001,Aconchi,0110,82,0,...,0,1.21951219512195,0,56.0975609756098,0,87.8048780487805,1.21951219512195,121.872595208357,Bajo,0.952023017058022



DENUE Sonora 2020 — Resumen
Shape: (13794, 42)

Distribución de códigos SCIAN de interés:
  522110 |  1478 registros | Banca múltiple
  722515 |  1590 registros | Cafeterías, fuentes de sodas, neverías, refresquer
  531210 |   213 registros | Inmobiliarias y corredores de bienes raíces
  522292 |     0 registros | N/A
  468211 |     0 registros | N/A
  468212 |     0 registros | N/A


---
## 3. Variable Objetivo (Target)

### `VIVPAR_DES` — Viviendas Particulares Deshabitadas

Esta es nuestra **variable objetivo** para la fase supervisada y el indicador central de rezago en la fase de clustering.

**Justificación teórica:**  
Según el INEGI (Censo 2020), una vivienda "deshabitada" es aquella cuya puerta estuvo cerrada durante los tres visitas del censo, sin señales de ocupación permanente. Esta definición captura **abandono efectivo**, no uso temporal. El fenómeno de abandono masivo de vivienda en México —documentado por CONAVI y la SEDATU— está directamente vinculado a:
1. Pérdida de valor del suelo (rezago urbano).
2. Déficit de servicios y equipamiento.
3. Procesos de vaciamiento demográfico por migración.

**Para la Fase No Supervisada (K-Means):**  
Usaremos la *tasa de viviendas deshabitadas* (`VIVPAR_DES / TVIVPAR`) como variable de validación interna: esperamos que el cluster de "alto rezago" concentre los AGEBs con mayor tasa de abandono, sin haberla incluido directamente en el entrenamiento.

**Para la Fase Supervisada (Random Forest / Gradient Boosting):**  
`VIVPAR_DES` se usará como target en dos formulaciones:
- **Regresión:** predecir el conteo absoluto o la tasa de abandono.
- **Clasificación:** binarizar con un umbral (ej. tasa > percentil 75) para etiquetar AGEBs como "alto riesgo de abandono" vs "bajo riesgo".

> **Nota:** Para la regresión, aplicaremos transformación log1p para corregir la asimetría positiva (muchos AGEBs con pocas viviendas deshabitadas y pocos con muchísimas).

In [8]:
# Verificación del target: distribución de VIVPAR_DES
target_raw = pd.to_numeric(df_censo_6c['VIVPAR_DES'], errors='coerce')
tvivpar_raw = pd.to_numeric(df_censo_6c['TVIVPAR'], errors='coerce')

tasa_abandono = target_raw / tvivpar_raw.replace(0, np.nan)

print("Estadísticas de VIVPAR_DES (conteo absoluto):")
print(target_raw.describe().round(2))
print(f"\nValores nulos (asteriscos INEGI convertidos a NaN): {target_raw.isna().sum()}")
print(f"\nEstadísticas de TASA de abandono (VIVPAR_DES / TVIVPAR):")
print(tasa_abandono.describe().round(4))
print(f"\nPercentil 75 de tasa: {tasa_abandono.quantile(0.75):.4f}")
print(f"Umbral sugerido para clasificación binaria: tasa > {tasa_abandono.quantile(0.75):.4f}")

Estadísticas de VIVPAR_DES (conteo absoluto):
count    1587.00
mean       52.82
std        58.15
min         0.00
25%         9.00
50%        39.00
75%        76.00
max       759.00
Name: VIVPAR_DES, dtype: float64

Valores nulos (asteriscos INEGI convertidos a NaN): 230

Estadísticas de TASA de abandono (VIVPAR_DES / TVIVPAR):
count    1465.0000
mean        0.1420
std         0.1094
min         0.0000
25%         0.0832
50%         0.1154
75%         0.1667
max         0.9531
dtype: float64

Percentil 75 de tasa: 0.1667
Umbral sugerido para clasificación binaria: tasa > 0.1667


---
## 4. Selección de Features

### Marco Conceptual

El modelo predice **abandono de vivienda**, un fenómeno que emerge en dos extremos del espectro urbano:
- **Rezago:** zonas marginadas donde la pobreza, hacinamiento y falta de servicios expulsan a los habitantes.
- **Plusvalía / Gentrificación:** zonas donde el alza de valor del suelo desplaza a residentes de bajos ingresos.

Por esto, agrupamos los features en cuatro familias:

| Familia | Fuente | Señal |
|---|---|---|
| Indicadores de Rezago Habitacional | INEGI Censo | Hacinamiento, piso de tierra, sin servicios |
| Indicadores de Bienestar y TICs | INEGI Censo | Internet, autos, computadora |
| Presencia de Comercio de Plusvalía | DENUE | Bancos, cafés, inmobiliarias |
| Presencia de Comercio de Rezago | DENUE | Empeños, yonques, artículos usados |
| Índice Sintético de Marginación | CONAPO | Grado de marginación a nivel AGEB |

---
### 4.1 Features del Censo INEGI — Indicadores de Rezago Habitacional

**Variables seleccionadas y justificación:**

- **`VPH_PISOTI`** (Viviendas con piso de tierra): Indicador clásico de pobreza habitacional extrema. Alta correlación con abandono porque son viviendas sin condiciones mínimas de habitabilidad. Relevante para K-Means porque actúa como centroide diferenciador del cluster de rezago.

- **`VPH_NODREN`** (Sin drenaje) y **`VPH_S_ELEC`** (Sin electricidad): La ausencia de servicios básicos es el predictor más robusto de desvalorización del suelo. En modelos de árbol (Random Forest, Gradient Boosting), estas variables tienen alta importancia porque crean divisiones claras entre zonas servidas y no servidas.

- **`PRO_OCUP_C`** (Promedio de ocupantes por cuarto): Índice de hacinamiento. Cuando este valor > 2.5, la literatura urbana (CONAVI) lo clasifica como hacinamiento severo, precursor del abandono por deterioro de la convivencia.

- **`VPH_1CUART`** (Viviendas de un solo cuarto): Complementa el hacinamiento. Viviendas muy pequeñas en zonas periféricas son las primeras en ser abandonadas cuando la familia mejora económicamente y se muda.

- **`VPH_SNBIEN`** (Sin ningún bien): El hogar sin refrigerador, TV ni celular es el proxy socioeconómico más bajo disponible en el censo. Excelente separador para PCA porque captura varianza transversal de pobreza.

### 4.2 Features del Censo INEGI — Indicadores de Bienestar (Proxy de Plusvalía)

- **`VPH_INTER`** (Con Internet): Correlaciona positivamente con el nivel socioeconómico del AGEB. Las zonas con alta penetración de internet son justamente las que experimentan gentrificación y presión sobre el precio del suelo.

- **`VPH_AUTOM`** (Con automóvil): Proxy de movilidad e ingreso. Un AGEB con alta proporción de hogares con auto indica perfil socioeconómico medio-alto. Importante para K-Means porque diferencia el cluster de "plusvalía" del de "rezago".

- **`VPH_PC`** (Con computadora/laptop/tablet): Junto con Internet, forma un índice de conectividad digital. Gradient Boosting lo utilizará para capturar no-linealidades: hay zonas con internet pero sin auto (clase media urbana) que tienen dinámicas distintas a zonas con ambos.

- **`VPH_REFRI`** (Con refrigerador): Indicador de bienestar básico. A diferencia de los anteriores, este distingue entre pobreza extrema (sin refrigerador) y pobreza moderada o superior. Ayuda al modelo a estratificar los AGEBs intermedios.

- **`GRAPROES`** (Grado promedio de escolaridad): Proxy sociodemográfico clave. La literatura de gentrificación (Hammel & Wyly, 1996) establece que el nivel educativo del vecindario es predictor del desplazamiento. En Random Forest, este feature suele aparecer en el top-5 de importancia.

### 4.3 Features del DENUE — Densidad de Establecimientos por AGEB

La estrategia es **agregar el DENUE al nivel AGEB**: contar cuántos establecimientos de cada tipo SCIAN existen dentro de cada AGEB. El resultado es un conteo entero por AGEB que actúa como feature espacial.

**Codes SCIAN de Plusvalía:**
- `522110` — Banca múltiple: Los bancos se instalan en zonas con alta actividad económica y poder adquisitivo. Su presencia es un indicador líder de valorización.
- `722515` — Cafeterías y similares: Proxy del proceso de gentrificación (el "índice del café de especialidad" popularizado en estudios urbanos). La llegada de cafés de tercera ola precede el alza del valor del suelo.
- `531210` — Agencias inmobiliarias: Señal directa de un mercado activo de compraventa. Alta densidad indica zonas con demanda y plusvalía.

**Codes SCIAN de Rezago:**
- `522292` — Casas de empeño: Se concentran en colonias de bajos ingresos donde las familias usan sus bienes como garantía de crédito informal. Alta correlación con pobreza.
- `468211` — Tianguis de artículos usados: Mercados informales que indican economías de subsistencia y desvalorización del espacio público.
- `468212` — Yonques (deshuesaderos): Usos de suelo de baja rentabilidad asociados a zonas de abandono industrial/habitacional. Alta densidad predice deterioro del entorno.

In [9]:
# ── Definición de los features seleccionados ─────────────────────────────────

# Features del Censo INEGI (nombres de columna exactos)
FEATURES_REZAGO_CENSO = [
    'VPH_PISOTI',   # Piso de tierra
    'VPH_NODREN',   # Sin drenaje
    'VPH_S_ELEC',   # Sin electricidad
    'PRO_OCUP_C',   # Hacinamiento (ocupantes/cuarto)
    'VPH_1CUART',   # Viviendas de 1 solo cuarto
    'VPH_SNBIEN',   # Sin ningún bien
]

FEATURES_PLUSVALIA_CENSO = [
    'VPH_INTER',    # Con internet
    'VPH_AUTOM',    # Con automóvil
    'VPH_PC',       # Con computadora/tablet
    'VPH_REFRI',    # Con refrigerador
    'GRAPROES',     # Grado promedio de escolaridad
]

# Features del DENUE (se construirán como conteos por AGEB)
SCIAN_PLUSVALIA = {
    'n_bancos':      '522110',
    'n_cafes':       '722515',
    'n_inmobiliarias': '531210',
}

SCIAN_REZAGO = {
    'n_empenos':     '522292',
    'n_usados':      '468211',
    'n_yonques':     '468212',
}

# Variable objetivo
TARGET = 'VIVPAR_DES'
# Denominador para calcular tasa
DENOMINADOR = 'TVIVPAR'

# Columna de unión
JOIN_KEY = 'CVE_AGEB'

print("Features de Rezago (Censo):", FEATURES_REZAGO_CENSO)
print("\nFeatures de Plusvalía (Censo):", FEATURES_PLUSVALIA_CENSO)
print("\nFeatures DENUE Plusvalía:", list(SCIAN_PLUSVALIA.keys()))
print("Features DENUE Rezago:", list(SCIAN_REZAGO.keys()))
print(f"\nTarget: {TARGET}")
print(f"Join key: {JOIN_KEY}")

Features de Rezago (Censo): ['VPH_PISOTI', 'VPH_NODREN', 'VPH_S_ELEC', 'PRO_OCUP_C', 'VPH_1CUART', 'VPH_SNBIEN']

Features de Plusvalía (Censo): ['VPH_INTER', 'VPH_AUTOM', 'VPH_PC', 'VPH_REFRI', 'GRAPROES']

Features DENUE Plusvalía: ['n_bancos', 'n_cafes', 'n_inmobiliarias']
Features DENUE Rezago: ['n_empenos', 'n_usados', 'n_yonques']

Target: VIVPAR_DES
Join key: CVE_AGEB


### 4.4 Construcción de Features DENUE — Agregación por AGEB

Construimos la CVE_AGEB en el DENUE concatenando `cve_ent + cve_mun + cve_loc + ageb` para hacer el join con el Censo.

In [10]:
# Filtrar DENUE a los 6 municipios objetivo y construir CVE_AGEB
df_denue_6c = df_denue[
    df_denue['cve_mun'].str.zfill(3).isin(MUNICIPIOS_SONORA)
].copy()

df_denue_6c['CVE_AGEB'] = (
    df_denue_6c['cve_ent'].str.zfill(2) +
    df_denue_6c['cve_mun'].str.zfill(3) +
    df_denue_6c['cve_loc'].str.zfill(4) +
    df_denue_6c['ageb'].str.zfill(4)
)

print(f"DENUE filtrado a 6 ciudades: {df_denue_6c.shape[0]:,} unidades económicas")
print(f"AGEBs únicos en DENUE: {df_denue_6c['CVE_AGEB'].nunique():,}")

# ── Agregación por AGEB compatible con pandas 3.x ────────────────────────────
# Creamos una columna booleana por cada código SCIAN y luego sumamos por AGEB.
# Este patrón es más claro y evita el comportamiento cambiante de groupby.apply().
todos_scian = {**SCIAN_PLUSVALIA, **SCIAN_REZAGO}

df_denue_marcado = df_denue_6c[['CVE_AGEB', 'codigo_act']].copy()
for nombre, codigo in todos_scian.items():
    df_denue_marcado[nombre] = (df_denue_marcado['codigo_act'] == codigo).astype(int)

denue_pivot = (
    df_denue_marcado
    .groupby('CVE_AGEB')[list(todos_scian.keys())]
    .sum()
    .reset_index()
)

print(f"\nTabla DENUE agregada por AGEB:")
print(f"Shape: {denue_pivot.shape}")
display(denue_pivot.head())
print(f"\nAGEBs con al menos 1 banco:  {(denue_pivot['n_bancos'] > 0).sum()}")
print(f"AGEBs con al menos 1 empeño: {(denue_pivot['n_empenos'] > 0).sum()}")

DENUE filtrado a 6 ciudades: 13,444 unidades económicas
AGEBs únicos en DENUE: 1,140

Tabla DENUE agregada por AGEB:
Shape: (1140, 7)


,CVE_AGEB,n_bancos,n_cafes,n_inmobiliarias,n_empenos,n_usados,n_yonques
0,2601800010531,0,0,0,0,0,0
1,2601800010546,1,4,0,0,0,0
2,2601800010550,0,15,2,0,0,0
3,2601800010565,0,10,0,0,0,0
4,260180001057A,3,9,8,0,0,0



AGEBs con al menos 1 banco:  373
AGEBs con al menos 1 empeño: 0


---
## 5. Limpieza Inicial: Manejo de Valores Nulos Ocultos del INEGI

> ### Advertencia crítica para el Pipeline ETL
>
> El INEGI **no utiliza `NaN` estándar** para indicar datos faltantes. En cambio, usa dos tokens de texto:
> - `*` (asterisco): valor suprimido por confidencialidad estadística (cuando hay < 3 casos).
> - `N/D`: "No disponible", dato no captado o no aplicable para ese nivel geográfico.
>
> Si estos tokens se pasan directamente a scikit-learn, **cualquier operación numérica fallará silenciosamente** o lanzará un `ValueError`. Esto es especialmente peligroso en un pipeline automatizado porque el error puede no detectarse hasta el momento del entrenamiento.
>
> **Solución:** `pd.to_numeric(..., errors='coerce')` convierte todos los valores no-numéricos a `NaN` de forma vectorizada y explícita. Esto permite usar `SimpleImputer` o `KNNImputer` de scikit-learn en el siguiente paso del pipeline.

In [11]:
# Demostración del problema: valores crudos antes de limpiar
cols_muestra = FEATURES_REZAGO_CENSO + FEATURES_PLUSVALIA_CENSO + [TARGET]
df_raw_muestra = df_censo_6c[cols_muestra].head(20)

# Detectar tokens problemáticos del INEGI
tokens_inegi = ['*', 'N/D']
for col in cols_muestra:
    for token in tokens_inegi:
        n = (df_censo_6c[col] == token).sum()
        if n > 0:
            print(f"  [ALERTA] Columna '{col}' contiene {n} valores '{token}'")

  [ALERTA] Columna 'VPH_PISOTI' contiene 475 valores '*'
  [ALERTA] Columna 'VPH_NODREN' contiene 405 valores '*'
  [ALERTA] Columna 'VPH_S_ELEC' contiene 504 valores '*'
  [ALERTA] Columna 'PRO_OCUP_C' contiene 93 valores '*'
  [ALERTA] Columna 'VPH_1CUART' contiene 348 valores '*'
  [ALERTA] Columna 'VPH_SNBIEN' contiene 483 valores '*'
  [ALERTA] Columna 'VPH_INTER' contiene 168 valores '*'
  [ALERTA] Columna 'VPH_AUTOM' contiene 138 valores '*'
  [ALERTA] Columna 'VPH_PC' contiene 178 valores '*'
  [ALERTA] Columna 'VPH_REFRI' contiene 119 valores '*'
  [ALERTA] Columna 'GRAPROES' contiene 93 valores '*'
  [ALERTA] Columna 'VIVPAR_DES' contiene 230 valores '*'


In [12]:
# ── Limpieza correcta usando pd.to_numeric(errors='coerce') ──────────────────
# Seleccionar solo las columnas que usaremos + identificadores
TODAS_FEATURES = FEATURES_REZAGO_CENSO + FEATURES_PLUSVALIA_CENSO + [TARGET, DENOMINADOR]

df_limpio = df_censo_6c[['CVE_AGEB', 'NOM_MUN'] + TODAS_FEATURES].copy()

# Convertir todas las columnas numéricas (asteriscos y N/D → NaN)
for col in TODAS_FEATURES:
    df_limpio[col] = pd.to_numeric(df_limpio[col], errors='coerce')

# Reporte de nulos tras conversión
print("Reporte de NaN por columna tras conversión con errors='coerce':")
print("-" * 55)
nulos = df_limpio[TODAS_FEATURES].isna().sum()
pct_nulos = (nulos / len(df_limpio) * 100).round(2)
reporte = pd.DataFrame({'NaN_count': nulos, 'NaN_%': pct_nulos})
print(reporte.to_string())
print(f"\nTotal AGEBs: {len(df_limpio):,}")
print(f"AGEBs sin ningún nulo: {df_limpio[TODAS_FEATURES].dropna().shape[0]:,}")

Reporte de NaN por columna tras conversión con errors='coerce':
-------------------------------------------------------
            NaN_count  NaN_%
VPH_PISOTI        475  26.14
VPH_NODREN        405  22.29
VPH_S_ELEC        504  27.74
PRO_OCUP_C         93   5.12
VPH_1CUART        348  19.15
VPH_SNBIEN        483  26.58
VPH_INTER         168   9.25
VPH_AUTOM         138   7.59
VPH_PC            178   9.80
VPH_REFRI         119   6.55
GRAPROES           93   5.12
VIVPAR_DES        230  12.66
TVIVPAR           126   6.93

Total AGEBs: 1,817
AGEBs sin ningún nulo: 621


**Interpretación de los nulos:**

Los `NaN` que aparecen aquí provienen casi en su totalidad de dos fuentes:
1. **Asteriscos de confidencialidad (`*`):** AGEBs muy pequeñas donde el INEGI suprimió el dato para evitar identificar a individuos. La estrategia correcta para el pipeline ETL es **imputar con la mediana del municipio** (no la media, que es sensible a outliers urbanos).
2. **`N/D` de nivel geográfico:** Filas que corresponden a totales de localidad que no tienen datos a nivel de manzana. Estas filas ya fueron excluidas al filtrar `MZA == '000'` y `AGEB != '0000'`.

> **Decisión para el Pipeline :** Se usará `sklearn.impute.SimpleImputer(strategy='median')` agrupado por municipio. Esta estrategia preserva la heterogeneidad entre ciudades (Hermosillo ≠ Navojoa en indicadores de bienestar).

---
## 6. Construcción del Dataset Maestro (Pre-Pipeline)

Unimos Censo + DENUE en un solo DataFrame por CVE_AGEB. Este dataset será la entrada del Pipeline ETL en la siguiente fase.

In [13]:
# Join Censo limpio + DENUE pivotado
# LEFT JOIN: conservamos todos los AGEBs del censo; si no hay negocios, los conteos = 0
df_maestro = df_limpio.merge(denue_pivot, on='CVE_AGEB', how='left')

# Los AGEBs sin ningún establecimiento de interés tendrán NaN → rellenar con 0
cols_denue = list(todos_scian.keys())
df_maestro[cols_denue] = df_maestro[cols_denue].fillna(0).astype(int)

# Calcular tasa de abandono (feature derivado y alternativa de target para regresión)
df_maestro['tasa_abandono'] = (
    df_maestro[TARGET] / df_maestro[DENOMINADOR].replace(0, np.nan)
).round(4)

print(f"Dataset Maestro — Shape final: {df_maestro.shape}")
print(f"\nColumnas del dataset:")
for col in df_maestro.columns:
    print(f"  {col}")
print(f"\nPrimeras filas:")
display(df_maestro.head())

Dataset Maestro — Shape final: (1817, 22)

Columnas del dataset:
  CVE_AGEB
  NOM_MUN
  VPH_PISOTI
  VPH_NODREN
  VPH_S_ELEC
  PRO_OCUP_C
  VPH_1CUART
  VPH_SNBIEN
  VPH_INTER
  VPH_AUTOM
  VPH_PC
  VPH_REFRI
  GRAPROES
  VIVPAR_DES
  TVIVPAR
  n_bancos
  n_cafes
  n_inmobiliarias
  n_empenos
  n_usados
  n_yonques
  tasa_abandono

Primeras filas:


,CVE_AGEB,NOM_MUN,VPH_PISOTI,VPH_NODREN,VPH_S_ELEC,PRO_OCUP_C,VPH_1CUART,VPH_SNBIEN,VPH_INTER,VPH_AUTOM,...,GRAPROES,VIVPAR_DES,TVIVPAR,n_bancos,n_cafes,n_inmobiliarias,n_empenos,n_usados,n_yonques,tasa_abandono
0,2601800010531,Cajeme,9.0,7.0,5.0,0.77,31.0,7.0,656.0,604.0,...,10.07,199.0,1324.0,0,0,0,0,0,0,0.1503
1,2601800010546,Cajeme,0.0,0.0,0.0,0.52,3.0,0.0,477.0,460.0,...,12.86,50.0,611.0,1,4,0,0,0,0,0.0818
2,2601800010550,Cajeme,NaN,0.0,0.0,0.51,3.0,0.0,601.0,594.0,...,13.57,113.0,773.0,0,15,2,0,0,0,0.1462
3,2601800010565,Cajeme,4.0,0.0,0.0,0.47,3.0,0.0,446.0,462.0,...,13.99,51.0,519.0,0,10,0,0,0,0,0.0983
4,260180001057A,Cajeme,NaN,0.0,0.0,0.47,0.0,0.0,354.0,354.0,...,13.98,39.0,369.0,3,9,8,0,0,0,0.1057


In [14]:
# Estadísticas descriptivas del dataset maestro
print("Estadísticas descriptivas del Dataset Maestro:")
TODAS_FEATURES_FINAL = (
    FEATURES_REZAGO_CENSO + FEATURES_PLUSVALIA_CENSO +
    cols_denue + [TARGET, 'tasa_abandono']
)
display(df_maestro[TODAS_FEATURES_FINAL].describe().round(3))

Estadísticas descriptivas del Dataset Maestro:


,VPH_PISOTI,VPH_NODREN,VPH_S_ELEC,PRO_OCUP_C,VPH_1CUART,VPH_SNBIEN,VPH_INTER,VPH_AUTOM,VPH_PC,VPH_REFRI,GRAPROES,n_bancos,n_cafes,n_inmobiliarias,n_empenos,n_usados,n_yonques,VIVPAR_DES,tasa_abandono
count,1342.000,1412.000,1313.000,1724.000,1469.000,1334.000,1649.000,1679.000,1639.000,1698.000,1724.000,1817.000,1817.000,1817.000,1817.0,1817.0,1817.0,1587.000,1465.000
mean,7.756,5.411,1.892,0.898,12.950,1.978,249.814,240.231,183.821,343.792,9.709,0.756,0.851,0.114,0.0,0.0,0.0,52.818,0.142
std,21.625,27.547,3.790,0.412,24.265,4.457,303.676,287.277,238.153,406.120,3.625,2.580,1.839,0.555,0.0,0.0,0.0,58.151,0.109
min,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0,0.0,0.000,0.000
25%,0.000,0.000,0.000,0.698,0.000,0.000,21.000,22.500,14.000,35.000,8.780,0.000,0.000,0.000,0.0,0.0,0.0,9.000,0.083
50%,3.000,0.000,0.000,0.910,5.000,0.000,160.000,155.000,110.000,229.500,10.055,0.000,0.000,0.000,0.0,0.0,0.0,39.000,0.115
75%,8.000,3.000,3.000,1.110,16.000,3.000,377.000,363.000,268.500,516.750,11.612,0.000,1.000,0.000,0.0,0.0,0.0,76.000,0.167
max,378.000,502.000,60.000,3.000,326.000,61.000,3561.000,3608.000,2694.000,6042.000,16.920,39.000,21.000,8.000,0.0,0.0,0.0,759.000,0.953


---
## 7. Resumen de Decisiones y Entregables para las Fases Siguientes

### 7.1 Mapa de Features Seleccionados

| # | Nombre | Fuente | Tipo | Señal | Rol en modelo |
|---|--------|--------|------|-------|---------------|
| 1 | `VPH_PISOTI` | Censo INEGI | Numérico | Rezago | Feature K-Means + RF |
| 2 | `VPH_NODREN` | Censo INEGI | Numérico | Rezago | Feature K-Means + RF |
| 3 | `VPH_S_ELEC` | Censo INEGI | Numérico | Rezago | Feature K-Means + RF |
| 4 | `PRO_OCUP_C` | Censo INEGI | Numérico | Rezago | Feature K-Means + RF |
| 5 | `VPH_1CUART` | Censo INEGI | Numérico | Rezago | Feature K-Means + RF |
| 6 | `VPH_SNBIEN` | Censo INEGI | Numérico | Rezago | Feature K-Means + RF |
| 7 | `VPH_INTER` | Censo INEGI | Numérico | Plusvalía | Feature K-Means + RF |
| 8 | `VPH_AUTOM` | Censo INEGI | Numérico | Plusvalía | Feature K-Means + RF |
| 9 | `VPH_PC` | Censo INEGI | Numérico | Plusvalía | Feature K-Means + RF |
| 10 | `VPH_REFRI` | Censo INEGI | Numérico | Plusvalía | Feature K-Means + RF |
| 11 | `GRAPROES` | Censo INEGI | Numérico | Plusvalía | Feature K-Means + RF |
| 12 | `n_bancos` | DENUE | Conteo | Plusvalía | Feature RF/GBM |
| 13 | `n_cafes` | DENUE | Conteo | Plusvalía | Feature RF/GBM |
| 14 | `n_inmobiliarias` | DENUE | Conteo | Plusvalía | Feature RF/GBM |
| 15 | `n_empenos` | DENUE | Conteo | Rezago | Feature RF/GBM |
| 16 | `n_usados` | DENUE | Conteo | Rezago | Feature RF/GBM |
| 17 | `n_yonques` | DENUE | Conteo | Rezago | Feature RF/GBM |
| — | `VIVPAR_DES` | Censo INEGI | Numérico | **TARGET** | Variable objetivo |
| — | `tasa_abandono` | Derivado | Float [0,1] | **TARGET** | Target alternativo (regresión) |

### 7.2 Checklist para el Pipeline ETL 

- [ ] Normalizar features con `StandardScaler` antes de K-Means y PCA (los conteos DENUE y los conteos VPH tienen escalas muy distintas).
- [ ] Imputar NaN con `SimpleImputer(strategy='median')` por municipio.
- [ ] Para clasificación: crear columna `abandono_alto` = `int(tasa_abandono > percentil_75)`.
- [ ] Para regresión: aplicar `np.log1p(VIVPAR_DES)` como target transformado.
- [ ] Validar que `CVE_AGEB` sea siempre string de 13 caracteres antes del join.
- [ ] Registrar experimentos en MLflow: incluir `feature_names`, `n_features`, `n_samples` como parámetros.

In [15]:
# ── Exportar lista canónica de features para uso en el pipeline ───────────────
import json

feature_schema = {
    "join_key":           JOIN_KEY,
    "target":             TARGET,
    "target_rate":        "tasa_abandono",
    "denominator":        DENOMINADOR,
    "features_rezago":    FEATURES_REZAGO_CENSO,
    "features_plusvalia": FEATURES_PLUSVALIA_CENSO,
    "features_denue":     list(todos_scian.keys()),
    "scian_codes":        todos_scian,
    "municipios":         MUNICIPIOS_SONORA,
    "n_features_total":   len(FEATURES_REZAGO_CENSO) + len(FEATURES_PLUSVALIA_CENSO) + len(todos_scian),
}

print("Schema de features (para pasar a MLflow como parámetros):")
print(json.dumps(feature_schema, indent=2, ensure_ascii=False))

print(f"\nTotal de features de entrada al modelo: {feature_schema['n_features_total']}")
print(f"Total de AGEBs en dataset maestro: {len(df_maestro):,}")

Schema de features (para pasar a MLflow como parámetros):
{
  "join_key": "CVE_AGEB",
  "target": "VIVPAR_DES",
  "target_rate": "tasa_abandono",
  "denominator": "TVIVPAR",
  "features_rezago": [
    "VPH_PISOTI",
    "VPH_NODREN",
    "VPH_S_ELEC",
    "PRO_OCUP_C",
    "VPH_1CUART",
    "VPH_SNBIEN"
  ],
  "features_plusvalia": [
    "VPH_INTER",
    "VPH_AUTOM",
    "VPH_PC",
    "VPH_REFRI",
    "GRAPROES"
  ],
  "features_denue": [
    "n_bancos",
    "n_cafes",
    "n_inmobiliarias",
    "n_empenos",
    "n_usados",
    "n_yonques"
  ],
  "scian_codes": {
    "n_bancos": "522110",
    "n_cafes": "722515",
    "n_inmobiliarias": "531210",
    "n_empenos": "522292",
    "n_usados": "468211",
    "n_yonques": "468212"
  },
  "municipios": [
    "018",
    "029",
    "030",
    "042",
    "043",
    "055"
  ],
  "n_features_total": 17
}

Total de features de entrada al modelo: 17
Total de AGEBs en dataset maestro: 1,817
